# FATFORMER-XLA: PIPELINE TRÍCH XUẤT VÀ ĐÓNG GÓI COMMUNITYFORENSICS-EVAL (CVPR 2025)
### BỘ KIỂM ĐỊNH NGOẠI VI ĐỘC LẬP (UNSEEN EXTERNAL BENCHMARK — 2.000 ẢNH)

> **Mục tiêu**: Thu thập 2.000 ảnh từ `OwensLab/CommunityForensics-Eval` qua cơ chế Streaming Parquet (không tốn đĩa Colab).
> **Phân tầng 5 họ bộ sinh**: `1_commercial`, `2_latent_diffusion`, `3_pixel_diffusion`, `4_modern_gans`, `5_other_autoregressive`.
> **Bảo toàn Zero Leakage**: Kiểm toán cách ly SHA-256 đối chiếu `data_manifest.json` để loại trừ trùng lặp.
> **Lưu trữ**: Đóng gói thành `test_communityforensics.tar` (PNG Lossless) lưu tại `MyDrive/Fatformer/datasets/`.

### Bước 1: Kết nối Google Drive và Kiểm tra Dung Lượng Khả Dụng

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!df -h /content | tail -1   # Yêu cầu dung lượng trống >= 10 GB

### Bước 2: Cài Đặt Thư Viện Cần Thiết
Cài đặt thư viện `datasets` của Hugging Face để nạp dữ liệu dạng streaming.

In [ ]:
!pip install -q datasets "pillow<13.0,>=8.0"

### Bước 3: Trích Xuất Phân Tầng 2.000 Ảnh (Đã Fix Toàn Bộ Schema)
- **Split**: `CompEval` (chứa 51.836 ảnh).
- **Byte thô**: `image_data` qua `io.BytesIO` chuyển sang PIL Image.
- **Lọc kiến trúc**: Cột `architecture` ('Commercial', 'LatDiff', 'PixDiff', 'GAN', 'other').
- **Lọc nguồn ảnh thật**: Cột `real_source` ('coco', 'ffhq', 'raise', 'laion').
- **Đa dạng hóa**: Tối đa 60 ảnh cho mỗi tên mô hình cụ thể `model_name`.

In [ ]:
import os, io, json, hashlib, gc, ctypes, random
from collections import defaultdict
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image
from tqdm import tqdm

# Bật cấu hình tự động thu hồi bộ nhớ C++ của Linux glibc
os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

def free_ram():
    """Hàm cưỡng bức giải phóng cả RAM Python, PyArrow C++ Pool và Linux Heap."""
    gc.collect()
    pa.default_memory_pool().release_unused()
    if libc is not None:
        try:
            libc.malloc_trim(0)
        except Exception:
            pass

TARGET_DIR = "/content/dataset_local/test_communityforensics"
EXISTING_MANIFEST = "/content/drive/MyDrive/Fatformer/manifests/data_manifest.json"
SEED = 42

os.makedirs(TARGET_DIR, exist_ok=True)

# 1. Nạp danh sách SHA-256 cũ để đảm bảo Zero Data Leakage
existing_hashes = set()
if os.path.exists(EXISTING_MANIFEST):
    with open(EXISTING_MANIFEST, "r", encoding="utf-8") as f:
        manifest_data = json.load(f)
        records = manifest_data.get("records", manifest_data) if isinstance(manifest_data, dict) else manifest_data
        for entry in records:
            if "sha256" in entry:
                existing_hashes.add(entry["sha256"])
    print(f"[*] Đã nạp {len(existing_hashes)} mã băm SHA-256 để kiểm toán cách ly.")
    del manifest_data, records
    free_ram()

# 2. Định nghĩa chỉ tiêu phân tầng 5 tập con (Cân bằng nhị phân 50/50)
QUOTAS = {
    "1_commercial": {
        "arch_key": "Commercial",
        "fake_max": 300,
        "real_quotas": {"coco": 150, "raise": 75, "laion": 75}
    },
    "2_latent_diffusion": {
        "arch_key": "LatDiff",
        "fake_max": 300,
        "real_quotas": {"coco": 150, "ffhq": 75, "laion": 75}
    },
    "3_pixel_diffusion": {
        "arch_key": "PixDiff",
        "fake_max": 200,
        "real_quotas": {"coco": 100, "raise": 50, "ffhq": 50}
    },
    "4_modern_gans": {
        "arch_key": "GAN",
        "fake_max": 100,
        "real_quotas": {"ffhq": 50, "coco": 50}
    },
    "5_other_autoregressive": {
        "arch_key": "other",
        "fake_max": 100,
        "real_quotas": {"raise": 50, "laion": 50}
    },
}

for group in QUOTAS:
    os.makedirs(f"{TARGET_DIR}/{group}/0_real", exist_ok=True)
    os.makedirs(f"{TARGET_DIR}/{group}/1_fake", exist_ok=True)

counts = {
    g: {
        "fake": 0,
        "real": {src: 0 for src in cfg["real_quotas"]},
        "models_collected": defaultdict(int)
    }
    for g, cfg in QUOTAS.items()
}

MAX_PER_SPECIFIC_MODEL = 60

# 3. Kết nối HfFileSystem lấy danh sách các shard Parquet của split CompEval
print("[*] Đang lấy danh sách Parquet shards từ OwensLab/CommunityForensics-Eval...")
fs = HfFileSystem()
shard_paths = sorted(fs.glob("datasets/OwensLab/CommunityForensics-Eval/**/*CompEval*.parquet"))
if not shard_paths:
    shard_paths = sorted(fs.glob("datasets/OwensLab/CommunityForensics-Eval/**/*.parquet"))

# Trộn ngẫu nhiên thứ tự các file Parquet với SEED=42 để lấy mẫu phân tán đều
rng = random.Random(SEED)
rng.shuffle(shard_paths)
print(f"[*] Tìm thấy {len(shard_paths)} Parquet shards. Bắt đầu lọc 2 Pha siêu tiết kiệm RAM...")

manifest_records = []
total_collected = 0
pbar = tqdm(total=2000, desc="Đang trích xuất phân tầng 2.000 ảnh")

def match_quota(label: int, arch: str, model_name: str, real_src: str):
    """Kiểm tra nhanh trên metadata xem dòng này có cần lấy hay không (Không tốn RAM ảnh)."""
    if label == 1:
        for g, cfg in QUOTAS.items():
            if arch.lower() == cfg["arch_key"].lower():
                if counts[g]["fake"] < cfg["fake_max"]:
                    if counts[g]["models_collected"][model_name] < MAX_PER_SPECIFIC_MODEL:
                        return g, None
    elif label == 0:
        for g, cfg in QUOTAS.items():
            for src_key, src_max in cfg["real_quotas"].items():
                if src_key in real_src and counts[g]["real"][src_key] < src_max:
                    return g, src_key
    return None, None

# 4. Duyệt từng Shard và từng Row-Group theo cơ chế Metadata-First
for shard_path in shard_paths:
    if total_collected >= 2000:
        break

    try:
        with fs.open(shard_path, "rb") as f:
            pf = pq.ParquetFile(f)
            num_row_groups = pf.num_row_groups

            # Trộn thứ tự row_group trong shard
            rg_indices = list(range(num_row_groups))
            rng.shuffle(rg_indices)

            for rg_idx in rg_indices:
                if total_collected >= 2000:
                    break

                # PHA 1: CHỈ đọc 4 cột chữ (Metadata) của Row-Group này (~50 KB RAM!)
                meta_table = pf.read_row_group(
                    rg_idx,
                    columns=["label", "architecture", "model_name", "real_source"]
                )
                labels = meta_table["label"].to_pylist()
                archs = meta_table["architecture"].to_pylist()
                mnames = meta_table["model_name"].to_pylist()
                rsrcs = meta_table["real_source"].to_pylist()
                del meta_table

                # Kiểm tra trước xem những vị trí (row_idx) nào trong Row-Group này cần lấy
                candidate_rows = set()
                for i in range(len(labels)):
                    lbl = int(labels[i]) if labels[i] is not None else -1
                    arc = str(archs[i] or "").strip()
                    mnm = str(mnames[i] or "unknown").strip()
                    rsr = str(rsrcs[i] or "").lower().strip()
                    g_cand, _ = match_quota(lbl, arc, mnm, rsr)
                    if g_cand is not None:
                        candidate_rows.add(i)

                # Nếu cả Row-Group không có ảnh nào ta cần -> BỎ QUA NGAY (0 byte ảnh tải về!)
                if not candidate_rows:
                    continue

                # PHA 2: Chỉ khi có ảnh cần lấy, đọc image_data theo lô siêu nhỏ (batch_size=8 ảnh)
                row_offset = 0
                for batch in pf.iter_batches(batch_size=8, row_groups=[rg_idx], columns=["image_data"]):
                    img_col = batch.column("image_data")
                    batch_len = len(batch)

                    for b_i in range(batch_len):
                        cur_idx = row_offset + b_i
                        if cur_idx not in candidate_rows:
                            continue

                        lbl = int(labels[cur_idx]) if labels[cur_idx] is not None else -1
                        arc = str(archs[cur_idx] or "").strip()
                        mnm = str(mnames[cur_idx] or "unknown").strip()
                        rsr = str(rsrcs[cur_idx] or "").lower().strip()

                        # Kiểm tra lại quota tại thời điểm thực tế
                        assigned_group, matched_r_src = match_quota(lbl, arc, mnm, rsr)
                        if not assigned_group:
                            continue

                        raw_bytes = img_col[b_i].as_py()
                        if not raw_bytes:
                            continue

                        # Tính SHA-256 chỉ cho những ảnh ĐÃ ĐƯỢC CHỌN
                        img_hash = hashlib.sha256(raw_bytes).hexdigest()
                        if img_hash in existing_hashes:
                            del raw_bytes
                            continue

                        try:
                            with Image.open(io.BytesIO(raw_bytes)) as pil_img:
                                img = pil_img.convert("RGB")
                        except Exception:
                            del raw_bytes
                            continue
                        del raw_bytes

                        if lbl == 1:
                            idx = counts[assigned_group]["fake"]
                            safe_mname = mnm.replace("/", "_").replace(" ", "_")[:30]
                            fname = f"fake_{idx:04d}_{safe_mname}.png"
                            rel_path = f"{assigned_group}/1_fake/{fname}"
                            counts[assigned_group]["fake"] += 1
                            counts[assigned_group]["models_collected"][mnm] += 1
                        else:
                            idx = sum(counts[assigned_group]["real"].values())
                            fname = f"real_{idx:04d}_{matched_r_src}.png"
                            rel_path = f"{assigned_group}/0_real/{fname}"
                            counts[assigned_group]["real"][matched_r_src] += 1

                        save_path = os.path.join(TARGET_DIR, rel_path)
                        # compress_level=1 giữ nguyên Lossless PNG 100% nhưng giảm 70% RAM đệm zlib
                        img.save(save_path, format="PNG", compress_level=1)
                        del img
                        existing_hashes.add(img_hash)

                        manifest_records.append({
                            "rel_path": rel_path,
                            "group": assigned_group,
                            "label": lbl,
                            "architecture": arc,
                            "model_name": mnm,
                            "real_source": rsr,
                            "sha256": img_hash
                        })

                        total_collected += 1
                        pbar.update(1)

                        if total_collected >= 2000:
                            break

                    row_offset += batch_len
                    del img_col, batch
                    free_ram()  # Xả ngay PyArrow C++ Pool và Linux Heap sau mỗi lô 8 ảnh!

                    if total_collected >= 2000:
                        break

                del labels, archs, mnames, rsrcs
                free_ram()
    except Exception as e:
        print(f"\n[!] Bỏ qua shard lỗi mạng tạm thời ({shard_path}): {e}")
        free_ram()
        continue

pbar.close()
free_ram()

# Lưu manifest kiểm toán bên trong thư mục
manifest_out = os.path.join(TARGET_DIR, "cfe_manifest.json")
with open(manifest_out, "w", encoding="utf-8") as f:
    json.dump(manifest_records, f, indent=2, ensure_ascii=False)

print(f"\n[✓] Đã thu thập đủ {total_collected}/2.000 ảnh chuẩn hóa! RAM CPU an toàn tuyệt đối.")
for g, c in counts.items():
    print(f"  • {g:<25}: Fake={c['fake']} | Real={dict(c['real'])}")

### Bước 4: Đóng Gói Thành Tệp .TAR Đẩy Lên Google Drive
Sử dụng chuẩn `PAX_FORMAT` để tương thích 100% với môi trường Linux / Colab / Kaggle.

In [ ]:
import shutil, tarfile, os

LOCAL_TAR_PATH = "/content/test_communityforensics.tar"
DRIVE_TAR_PATH = "/content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar"
os.makedirs(os.path.dirname(DRIVE_TAR_PATH), exist_ok=True)

print(f"[*] Bước 4.1: Đang đóng gói nguyên tử trên ổ SSD cục bộ ({LOCAL_TAR_PATH})...")
with tarfile.open(LOCAL_TAR_PATH, "w", format=tarfile.PAX_FORMAT) as tar:
    tar.add(TARGET_DIR, arcname="test_communityforensics")

tar_sz_gb = os.path.getsize(LOCAL_TAR_PATH) / (1024 ** 3)
print(f"[✓] Đóng gói SSD hoàn tất ({tar_sz_gb:.2f} GB). Đang đồng bộ sang Google Drive 5TB...")

shutil.copyfile(LOCAL_TAR_PATH, DRIVE_TAR_PATH)
os.remove(LOCAL_TAR_PATH)  # Xóa bản tạm trên SSD sau khi đã lên Drive an toàn
free_ram()

print(f"[✓] HOÀN TẤT 100%! Vị trí lưu trữ: {DRIVE_TAR_PATH}")

### Bước 5: Chạy Đánh Giá Kiểm Chuẩn Tự Động Với `tools/eval_communityforensics.py`
Chạy 1 lệnh duy nhất để đo đạc và in ra Bảng A (Clean) và Bảng B (JPEG Q=30).

In [ ]:
# Đảm bảo clone repo và đứng tại thư mục gốc fatformer-xla
# %cd /content/fatformer-xla

!python tools/eval_communityforensics.py \
    --data_dir /content/dataset_local/test_communityforensics \
    --output_dir /content/drive/MyDrive/Fatformer/logs/cfe_benchmark